# Matching Methods

## Overview

Matching creates a comparison group by pairing each treated unit with a similar control unit, reducing confounding by making the groups more comparable on observed covariates.

**Matching approaches:**

| Method | Match on | Pros | Cons |
|---|---|---|---|
| Exact matching | Covariate values | No bias on matched vars | Fails with many covariates |
| Nearest-neighbour | Mahalanobis distance | Flexible | Curse of dimensionality |
| Propensity score matching | P(T=1|X) | Dimension reduction | PS model must be correct |
| Caliper matching | PS within tolerance | Prevents bad matches | May discard units |

After matching, check balance: standardised mean differences (SMD) should be < 0.1 for all covariates. Then estimate the ATT (Average Treatment effect on the Treated) from matched pairs.

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from scipy import stats
import statsmodels.formula.api as smf

rng = np.random.default_rng(42)
n = 600
elevation   = rng.uniform(50, 350, n)
nitrate     = rng.gamma(2, 2, n)
upstream    = rng.binomial(1, 0.4, n)
elev_s      = (elevation - elevation.mean())/elevation.std()
nitr_s      = (nitrate   - nitrate.mean())  /nitrate.std()
p_treat     = 1/(1+np.exp(-0.9*elev_s + 0.4*nitr_s - 0.3*upstream))
treatment   = rng.binomial(1, p_treat, n)
Y0 = 20 - 0.04*elevation - 0.6*nitrate + 1.5*upstream + rng.normal(0,3,n)
Y1 = Y0 + 3.5
Y_obs = np.where(treatment==1, Y1, Y0)
true_ATT = (Y1 - Y0)[treatment==1].mean()
df = pd.DataFrame({"elevation":elevation,"nitrate":nitrate,"upstream":upstream,
                    "treatment":treatment,"richness":Y_obs})
print(f"n={n}, treated={treatment.sum()}, control={n-treatment.sum()}")
print(f"True ATT: {true_ATT:.3f}")
print(f"Naive diff: {df.groupby('treatment')['richness'].mean().diff().iloc[-1]:.3f}")

---
## Propensity Score Estimation

In [ ]:
covs = ["elevation","nitrate","upstream"]
X_cov = df[covs]
scaler = StandardScaler().fit(X_cov)
X_sc = scaler.transform(X_cov)
ps_model = LogisticRegression(C=1.0, max_iter=500).fit(X_sc, df["treatment"])
df["ps"] = ps_model.predict_proba(X_sc)[:,1]
# Check balance before matching
def smd(x, treat):
    m1 = x[treat==1].mean(); m0 = x[treat==0].mean()
    s1 = x[treat==1].std();  s0 = x[treat==0].std()
    return (m1-m0) / np.sqrt((s1**2+s0**2)/2)

print("Standardised Mean Differences before matching (|SMD| < 0.1 = good balance):")
for cov in covs + ["ps"]:
    s = smd(df[cov].values, df["treatment"].values)
    flag = "***" if abs(s) > 0.1 else ""
    print(f"  {cov:12s}: SMD={s:.3f} {flag}")

---
## 1:1 Nearest-Neighbour Propensity Score Matching

In [ ]:
treated_idx   = df[df["treatment"]==1].index.tolist()
control_idx   = df[df["treatment"]==0].index.tolist()
ps_treated = df.loc[treated_idx,"ps"].values.reshape(-1,1)
ps_control = df.loc[control_idx,"ps"].values.reshape(-1,1)
# Nearest neighbour on PS with caliper 0.2*SD(PS)
caliper = 0.2 * df["ps"].std()
nn = NearestNeighbors(n_neighbors=1).fit(ps_control)
dists, indices = nn.kneighbors(ps_treated)
matched_treated = []
matched_control = []
for i, (dist, idx) in enumerate(zip(dists[:,0], indices[:,0])):
    if dist <= caliper:
        matched_treated.append(treated_idx[i])
        matched_control.append(control_idx[idx])
df_match = pd.concat([df.loc[matched_treated].assign(pair=range(len(matched_treated))),
                       df.loc[matched_control].assign(pair=range(len(matched_control)))])
print(f"Matched pairs: {len(matched_treated)} (caliper={caliper:.4f})")
print(f"  {len(treated_idx)-len(matched_treated)} treated units discarded (outside caliper)")
print("\nSMD after matching:")
for cov in covs + ["ps"]:
    s = smd(df_match[cov].values, df_match["treatment"].values)
    flag = "***" if abs(s) > 0.1 else "OK"
    print(f"  {cov:12s}: SMD={s:.3f}  {flag}")

---
## Balance Plot

In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
smds_before = [smd(df[c].values, df["treatment"].values) for c in covs+["ps"]]
smds_after  = [smd(df_match[c].values, df_match["treatment"].values) for c in covs+["ps"]]
y = np.arange(len(covs)+1)
ax.scatter(smds_before, y, s=80, color="steelblue", label="Before matching", zorder=3)
ax.scatter(smds_after,  y, s=80, color="#e74c3c",   label="After matching",  zorder=3)
ax.axvline(0,    color="black", lw=0.8)
ax.axvline( 0.1, color="grey", lw=1, linestyle="--", alpha=0.7)
ax.axvline(-0.1, color="grey", lw=1, linestyle="--", alpha=0.7)
ax.set_yticks(y); ax.set_yticklabels(covs+["propensity score"])
ax.set_xlabel("Standardised Mean Difference")
ax.set_title("Love Plot: Covariate Balance Before and After Matching")
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
# ATT estimation from matched pairs
matched_treated_df = df_match[df_match["treatment"]==1].reset_index(drop=True)
matched_control_df = df_match[df_match["treatment"]==0].reset_index(drop=True)
pair_diffs = matched_treated_df["richness"].values - matched_control_df["richness"].values
att_matched = pair_diffs.mean()
se_matched  = pair_diffs.std() / np.sqrt(len(pair_diffs))
ci = stats.t.ppf([0.025,0.975], len(pair_diffs)-1) * se_matched + att_matched
print(f"ATT (matched pairs): {att_matched:.3f} (SE={se_matched:.3f})")
print(f"95% CI: [{ci[0]:.3f}, {ci[1]:.3f}]")
print(f"True ATT: {true_ATT:.3f}")
print(f"\nNaive diff: {df.groupby('treatment')['richness'].mean().diff().iloc[-1]:.3f}")
print("Matching reduces confounding bias substantially")

---

## Common Pitfalls

**1. Estimating the ATE rather than ATT from matched data**  
Propensity score matching discards unmatched control units, which changes the estimand. The resulting estimate is the ATT (Average Treatment effect on the Treated), not the ATE for the full population. Interpret and report accordingly.

**2. Not checking balance after matching**  
The goal of matching is covariate balance, not the match itself. Always compute SMDs for all covariates after matching and produce a Love plot. An SMD > 0.1 after matching indicates residual imbalance that needs further adjustment.

**3. Using too large a caliper or no caliper**  
Without a caliper, even distant propensity score matches are accepted, producing poorly matched pairs. Use a caliper of 0.2 × SD(propensity score) as the standard starting point.

**4. Matching with replacement without accounting for it in variance estimation**  
When matching with replacement, control units may be used multiple times, reducing the effective sample size. Standard paired t-test SEs underestimate uncertainty. Use bootstrap SEs or weighted estimators when matching with replacement.

**5. Treating matching as a substitute for randomisation**  
Matching controls for observed confounders only. If an unmeasured confounder exists, matching does not eliminate bias. Always conduct a sensitivity analysis (e.g. Rosenbaum bounds) to assess how strong unmeasured confounding would need to be to overturn the conclusion.
---
*python_methods_library - Samantha McGarrigle*